# ci-proto — time-series forecasting on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/ci-proto-time-series-forecast-7e78/colab/ci-proto-time-series-forecast.ipynb)

[`ContinuousIntelligence/ci-proto`](https://huggingface.co/ContinuousIntelligence/ci-proto) is Continuous Intelligence's **167M time-series foundation model** for zero-shot probabilistic forecasting. It is not a language model: there is no tokenizer, no chat template, and no `generate()`. You pass numeric history (`[variables, time]`) and get nine quantile forecasts back (0.1, 0.2, …, 0.9). The Hub repo is **public** (not gated).

**Why this model.** Same job as the [Timestral-209M notebook](Timestral-209M-forecast.ipynb) — zero-shot quantile forecasts on a free T4 — but a different checkpoint and API (`pip install continuousintelligence`, then `ci.load_model`). The card also documents **multivariate** series, **past/future covariates**, timestamps, related series, and very long context: `model.ckpt` for histories up to **16,384** steps; `model-long.ckpt` for longer histories, selected automatically.

See [Use cases](https://github.com/donvito/notebooks/blob/melvin/ci-proto-time-series-forecast-7e78/docs/ci-proto-time-series-forecast.md) for when this 167M model is a good fit and when it is not.

**Licence (read this):** Hub tag is `other`. Code and weights use the [ci-proto Code and Model Licence v1.0](https://huggingface.co/ContinuousIntelligence/ci-proto/blob/main/LICENCE). It is **Apache-2.0 based**, plus **Section 10**: redistribution and externally available products (including another AI model trained or distilled from these weights or outputs) need a public attribution line and a **one-time** email to `contact@continuousintelligence.co.uk`. **Purely internal use is exempt** from Section 10. This notebook is a local Colab walkthrough, not a redistribution of the weights.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15 GB of VRAM. `model.ckpt` is about **669 MB** of weights (~167M parameters), so **CPU also works**; this notebook prefers CUDA when a GPU is attached. Fused Mamba-3 kernels in the package README need **Ampere-or-newer** GPUs. A T4 is Turing (`sm_75`), so you should see `model.backend == "torch"` (portable PyTorch). The released `tta.json` sets `bfloat16: true`; T4 has no native bfloat16, and PyTorch autocast then runs in float32.

**Links:** [model card](https://huggingface.co/ContinuousIntelligence/ci-proto) · [package README](https://github.com/continuousintelligence/ci) · [walkthrough](https://github.com/continuousintelligence/ci/blob/main/examples/walkthrough.ipynb) · [licence](https://huggingface.co/ContinuousIntelligence/ci-proto/blob/main/LICENCE) · [announcement](https://x.com/Continu_Intel/status/2120852944056709139) · [Timestral-209M](Timestral-209M-forecast.ipynb)


## 1. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. Colab already ships a CUDA torch; this cell imports **torch only**. A T4 is preferred. CPU is enough for this 167M checkpoint if no GPU is attached.


In [ ]:
import os

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    major, minor = torch.cuda.get_device_capability()
    free_b, total_b = torch.cuda.mem_get_info()
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
    print(f"capability {major}.{minor} | multiprocessors {props.multi_processor_count}")
    if major < 8:
        print("Turing / pre-Ampere GPU: ci-proto fused Mamba-3 is skipped; portable torch backend.")
else:
    print("device: CPU (fine for this checkpoint; a T4 is faster)")
    print("Select Runtime > Change runtime type > T4 GPU if you want the GPU path.")


def print_vram(prefix=""):
    if not torch.cuda.is_available():
        print(f"{prefix}CPU runtime — no VRAM stats")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"{prefix}VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


## 2. Install `continuousintelligence` (keep Colab's torch)

The card install is `pip install continuousintelligence` (Python 3.10–3.14). On Linux x86-64 that also pulls `cuda-toolkit[nvcc,cccl]==13.0.3`, `tilelang==0.1.8`, and `apache-tvm-ffi==0.1.9` for fused Mamba-3 kernels, and it can replace Colab's CUDA torch. A T4 cannot use those fused kernels (Ampere-or-newer only), so this cell installs the **0.1.1** wheel with `--no-deps` and only the portable extras. Use **`%pip`**, not `python -I -m pip`. Isolated mode drops Colab's site-packages and breaks the preinstalled torch. Do **not** `pip install torch`. Do **not** install `flash-attn`.


In [ ]:
%pip install -q --no-deps "continuousintelligence==0.1.1"
%pip install -q "einops>=0.8,<1" "huggingface-hub>=1,<3" "pandas>=2.2,<4" "matplotlib>=3.8,<4"

# Card command is: pip install continuousintelligence
# --no-deps keeps Colab's CUDA torch and skips cuda-toolkit 13 / tilelang / apache-tvm-ffi
# (fused extras; T4 is not Ampere). 0.1.1 is the 2026-11-12 PyPI release.
# Remaining deps from the wheel: einops, huggingface-hub, pandas, matplotlib.
# numpy and torch already satisfy the floor (numpy>=1.26, torch>=2.7).


> **Runtime tip:** pick a T4 if you have the choice. If `continuousintelligence` was imported before the install cell, use *Runtime > Restart session* and continue from the next cell. A fresh session does not need a restart.


## 3. Load ci-proto

`ci.load_model("ContinuousIntelligence/ci-proto")` downloads `config.json`, `tta.json`, and `model.ckpt` (~669 MB) into the Hub cache on the first run. `device="auto"` uses CUDA when it is available. `model-long.ckpt` is not loaded until a history is longer than 16,384 steps. Forecast outputs are always CPU `float32` tensors shaped `[variables, quantiles, horizon]`.


In [ ]:
import warnings

import continuousintelligence as ci

# Official walkthrough: CPU/T4 autocast can warn when tta.json sets bfloat16.
warnings.filterwarnings("ignore", message="Mismatch dtype between input and weight")

MODEL_ID = "ContinuousIntelligence/ci-proto"

model = ci.load_model(MODEL_ID, device="auto")
info = model.info or {}

print("repo:", MODEL_ID)
print("info:", info)
print("parameters (card):", info.get("parameters"))
print("quantiles:", [round(q, 1) for q in model.quantiles.tolist()])
print("device:", model.device)
print("backend:", model.backend, "(mamba3 = fused CUDA; torch = portable)")
print_vram("after load  ")


## 4. Bundled bike-rental example

The package ships five days of hourly bicycle rentals in Washington, D.C. (`ci.load_example()` → `[1, 120]`). Forecast the next 24 hours and draw the median plus nested quantile bands. Example data: Hadi Fanaee-T (2013), [Bike Sharing, UCI](https://doi.org/10.24432/C5W894), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).


In [ ]:
from importlib.resources import files

import matplotlib.pyplot as plt
import pandas as pd
import torch

history = ci.load_example()
print("history:", tuple(history.shape), history.dtype)

prediction = model.forecast(history, horizon=24)
print("prediction:", tuple(prediction.shape), "(variables, quantiles, horizon)")
print("median[:8]:", prediction[0, 4, :8].tolist())

csv_path = files("continuousintelligence").joinpath("_data/bike_sharing.csv")
frame = pd.read_csv(csv_path, parse_dates=["timestamp"])
hist_times = frame.loc[frame["split"].eq("history"), "timestamp"]
fut_times = frame.loc[frame["split"].eq("forecast"), "timestamp"]
all_times = pd.concat([hist_times, fut_times], ignore_index=True)
stamps_ns = torch.tensor(all_times.to_numpy(dtype="datetime64[ns]").astype("int64"), dtype=torch.int64)

fig = ci.plot_forecast(
    history,
    prediction,
    model.quantiles,
    timestamps=all_times,
    ylabel="Bike rentals per hour",
    output="forecast.png",
)
plt.show()
print("wrote forecast.png")
print_vram("after bike forecast  ")


## 5. Calendar timestamps and known-future covariates

From the [package README](https://github.com/continuousintelligence/ci#usage-details): `timestamps` is int64 epoch nanoseconds covering history **plus** horizon; `freq` (a pandas offset alias such as `"h"`) turns those stamps into calendar features. `future_covariates` is `[known_variables, T + H]` — only values known at forecast time. Boolean rows stay numeric; categorical rows are integer codes listed in `categorical_future`.

This cell reuses the bundled hourly grid. Working day is a 0/1 flag (weekend vs weekday). Hour of day is a 0–23 code marked categorical. Only `history` is forecast.


In [ ]:
workingday = torch.tensor((all_times.dt.day_of_week < 5).to_numpy(), dtype=torch.float32)
hour_of_day = torch.tensor(all_times.dt.hour.to_numpy(), dtype=torch.float32)
future_covariates = torch.stack([workingday, hour_of_day])
print("future_covariates:", tuple(future_covariates.shape), "(known, T+H)")

prediction_cov = model.forecast(
    history,
    horizon=24,
    future_covariates=future_covariates,
    categorical_future=(1,),
    timestamps=stamps_ns,
    freq="h",
)
print("prediction with covariates:", tuple(prediction_cov.shape))

q10, q50, q90 = 0, 4, 8
print(
    "median 10–90% width, history-only:",
    float((prediction[0, q90] - prediction[0, q10]).mean()),
)
print(
    "median 10–90% width, + covariates:",
    float((prediction_cov[0, q90] - prediction_cov[0, q10]).mean()),
)

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(hist_times, history[0].numpy(), color="#1f4e79", lw=1.3, label="history")
ax.plot(fut_times, prediction[0, q50].numpy(), color="#d62728", lw=1.6, label="history only")
ax.fill_between(
    fut_times,
    prediction[0, q10].numpy(),
    prediction[0, q90].numpy(),
    color="#d62728",
    alpha=0.12,
    label="history only 10–90%",
)
ax.plot(fut_times, prediction_cov[0, q50].numpy(), color="#2ca02c", lw=1.8, label="+ working day & hour")
ax.fill_between(
    fut_times,
    prediction_cov[0, q10].numpy(),
    prediction_cov[0, q90].numpy(),
    color="#2ca02c",
    alpha=0.18,
    label="+ covariates 10–90%",
)
ax.axvline(fut_times.iloc[0], color="#666666", ls="--", lw=1)
ax.set_title("ci-proto — history-only vs calendar covariates (same 24h horizon)")
ax.set_xlabel("timestamp")
ax.set_ylabel("rentals / hour")
ax.legend(loc="upper left", ncol=2, frameon=False)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()


## 6. Missing observations

The README: mark missing values as `NaN` in `history` (or any covariate). No imputation step. The cell below deletes a 12-hour block and every seventh hour, then forecasts the same 24-step horizon. Histories longer than 16,384 steps would auto-load `model-long.ckpt` (~669 MB extra); this notebook stays on the standard checkpoint.


In [ ]:
gappy = history.clone()
gappy[0, 36:48] = float("nan")
gappy[0, 6::7] = float("nan")
observed = float((~gappy.isnan()).float().mean())
print(f"observed: {observed:.0%} of the 120 hourly steps")

prediction_gap = model.forecast(gappy, horizon=24)
print("prediction with gaps:", tuple(prediction_gap.shape))
print("finite:", bool(torch.isfinite(prediction_gap).all()))

fig = ci.plot_forecast(
    gappy,
    prediction_gap,
    model.quantiles,
    timestamps=all_times,
    ylabel="Bike rentals per hour",
    output="forecast-missing.png",
)
plt.show()
print("wrote forecast-missing.png")
print_vram("after missing-data forecast  ")


## 7. Caveats and cleanup

**Licence.** [ci-proto Code and Model Licence v1.0](https://huggingface.co/ContinuousIntelligence/ci-proto/blob/main/LICENCE) (`license: other` on the Hub). Apache-2.0 based, plus Section 10 attribution and one-time notification for redistribution / externally available products. Internal use is exempt from Section 10. Keep the licence with any copy of the weights you distribute. The bundled bike CSV is [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Not an LLM.** Do not use ci-proto for chat, code, classification of text, RAG, or any other language-model task.

**Context.** Histories up to **16,384** steps use `model.ckpt`. Longer histories automatically use `model-long.ckpt` with the full supplied history (card: 2M+ context on that path). Horizon is any positive integer (package default 320). This notebook uses 24.

**Data shape.** `history` is `[V, T]`. Multivariate rows are forecast jointly unless you pass `series_ids`. `past_covariates` is `[past_variables, T]`; `future_covariates` is `[known_variables, T + H]`. Missing values are `NaN`.

**Quantile bands.** The nine knots are model quantiles, not a calibrated prediction interval for your series. A 10–90% band is not a promise that 80% of future points will land inside it.

**T4 / precision.** Fused `"mamba3"` needs Ampere-or-newer. T4 stays on `"torch"`. Released `tta.json` enables bfloat16 autocast; T4 has no native bf16. The package README: reduced-precision results can vary across runtimes. Reference results on the package README use H100 with PyTorch `2.13.0+cu130`.

**When not to rely on the demo numbers.** The plots are a smoke test on one bundled series. They are not a benchmark. Validate on your own history before inventory, staffing, or pricing decisions.

**Related notebook.** [`Timestral-209M-forecast.ipynb`](Timestral-209M-forecast.ipynb) is Mistral's 209M zero-shot forecaster (21 quantiles, Modified MIT).

The next cell drops the model so you can load something else on the same T4.


In [ ]:
import gc

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print_vram("after cleanup  ")
